In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Dataset path
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DataCoSupplyChainDataset.csv"
)

# Load the dataset
df = pd.read_csv(
    DATA_PATH,
    encoding="latin1"
)

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully!
Rows: 180,519
Columns: 53


In [3]:
# Step 9.1 — Create a working copy

df_clean = df.copy()

print("Original shape:", df.shape)
print("Working copy:", df_clean.shape)

Original shape: (180519, 53)
Working copy: (180519, 53)


In [5]:
# Convert common text placeholders into actual missing values

missing_tokens = ["", "NULL", "null", "N/A", "n/a"]

df_clean = df_clean.replace(
    missing_tokens,
    np.nan
)
# makes missing-value handling consistent, We won't fill missing values yet

In [9]:
duplicate_count = df_clean.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df_clean = df_clean.drop_dupliplicates()

print("Shape after duplicate check:", df_clean.shape)

Exact duplicate rows: 0
Shape after duplicate check: (180519, 53)


In [10]:
drop_cols = [
    # Completely empty or highly incomplete
    "Product Description",
    "Order Zipcode",

    # Personal information
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",
    "Customer Street",
    "Customer Zipcode",
    "Customer Id",
    "Order Customer Id",

    # Record and product identifiers
    "Order Id",
    "Order Item Id",
    "Order Item Cardprod Id",
    "Product Card Id",
    "Category Id",
    "Department Id",
    "Product Category Id",

    # Direct outcome leakage
    "Delivery Status",
    "Days for shipping (real)",
    "shipping date (DateOrders)",
    "Order Status",

    # Post-order or potentially outcome-derived fields
    "Benefit per order",
    "Sales per customer",
    "Order Item Profit Ratio",
    "Order Profit Per Order",

    # Redundant or unsuitable fields
    "Product Price",
    "Sales",
    "Order Item Total",
    "Product Image",
    "Product Name",
    "Product Status",

    # Raw coordinates: not needed for this first model
    "Latitude",
    "Longitude",
]

# Keep only columns that actually exist
drop_cols = [
    col for col in drop_cols
    if col in df_clean.columns
]

df_clean = df_clean.drop(columns=drop_cols)

print("Columns dropped:", len(drop_cols))
print("Remaining shape:", df_clean.shape)
print("Remaining columns:", df_clean.columns.tolist())

Columns dropped: 33
Remaining shape: (180519, 20)
Remaining columns: ['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Quantity', 'Order Region', 'Order State', 'Shipping Mode']


In [12]:
# The raw order date contains useful information, but a timestamp isn't always the most convenient input for our first model

# Parse order date
date_col = "order date (DateOrders)"

df_clean[date_col] = pd.to_datetime(
    df_clean[date_col],
    errors="coerce"
)

# Extract date features
df_clean["Order_Month"] = (
    df_clean[date_col].dt.month
)

df_clean["Order_DayOfWeek"] = (
    df_clean[date_col].dt.dayofweek
)

df_clean["Order_Day"] = (
    df_clean[date_col].dt.day
)

# Keep the original timestamp for chronological splitting
print(df_clean[date_col].min())
print(df_clean[date_col].max())

2015-01-01 00:00:00
2018-01-31 23:38:00


In [13]:
missing_after_drop = (
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_after_drop[
    missing_after_drop > 0
]

Series([], dtype: int64)

In [14]:
TARGET = "Late_delivery_risk"

print(df_clean[TARGET].value_counts(dropna=False))
print("Missing targets:", df_clean[TARGET].isna().sum())

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64
Missing targets: 0


In [15]:
df_clean = df_clean.dropna(
    subset=[TARGET]
).copy()

df_clean[TARGET] = df_clean[TARGET].astype(int)

assert set(df_clean[TARGET].unique()).issubset({0, 1})

print("Final cleaned shape:", df_clean.shape)

Final cleaned shape: (180519, 23)


In [16]:
feature_cols = [
    # Shipment configuration
    "Type",
    "Shipping Mode",
    "Days for shipment (scheduled)",

    # Order timing
    "Order_Month",
    "Order_DayOfWeek",
    "Order_Day",

    # Product and transaction details
    "Category Name",
    "Department Name",
    "Order Item Product Price",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Discount Rate",

    # Customer segment
    "Customer Segment",

    # Geographic and market context
    "Market",
    "Order Region",
    "Order Country",
    "Order State",
]

# Keep only features that exist in the cleaned dataset
feature_cols = [
    col for col in feature_cols
    if col in df_clean.columns
]

X = df_clean[feature_cols].copy()
y = df_clean[TARGET].copy()

print("Feature count:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

Feature count: 17
X shape: (180519, 17)
y shape: (180519,)


,Type,Shipping Mode,Days for shipment (scheduled),Order_Month,Order_DayOfWeek,Order_Day,Category Name,Department Name,Order Item Product Price,Order Item Quantity,Order Item Discount,Order Item Discount Rate,Customer Segment,Market,Order Region,Order Country,Order State
0,DEBIT,Standard Class,4,1,2,31,Sporting Goods,Fitness,327.75,1,13.110000,0.04,Consumer,Pacific Asia,Southeast Asia,Indonesia,Java Occidental
1,TRANSFER,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,16.389999,0.05,Consumer,Pacific Asia,South Asia,India,Rajastán
2,CASH,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,18.030001,0.06,Consumer,Pacific Asia,South Asia,India,Rajastán
3,DEBIT,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,22.940001,0.07,Home Office,Pacific Asia,Oceania,Australia,Queensland
4,PAYMENT,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,29.500000,0.09,Corporate,Pacific Asia,Oceania,Australia,Queensland


In [17]:
PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "supply_chain_cleaned.csv"
)

df_clean.to_csv(
    PROCESSED_PATH,
    index=False
)

print(f"Saved cleaned data to: {PROCESSED_PATH}")

Saved cleaned data to: /Users/Kushagra/Desktop/Code/LogiCast/data/processed/supply_chain_cleaned.csv


In [18]:
FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_columns.txt"
)

FEATURES_PATH.write_text(
    "\n".join(feature_cols)
)

print("Feature list saved.")

Feature list saved.


In [19]:
print("Cleaned dataset:", df_clean.shape)
print("Features:", X.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Missing target values:", y.isna().sum())
print("Target values:", sorted(y.unique()))

Cleaned dataset: (180519, 23)
Features: (180519, 17)
Missing values in X: 0
Missing target values: 0
Target values: [np.int64(0), np.int64(1)]
